# Araseの電磁場 despun データのチェック → DSI座標系の時点で擾乱が見えるかを確認。

# データ保存先

In [1]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# Araseの電場・磁場データのplot

In [3]:
import pyspedas as psp
import pytplot as pt

pt.del_data('*')

time_range = ['20220901/21:00:00', '20220902/00:00:00']

path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330'

psp.erg.pwe_efd(trange=time_range, level='l2', datatype='64', coord='dsi', no_update=True, get_support_data=True)
psp.erg.mgf(trange=time_range, level='l2', datatype='64hz', coord='dsi', no_update=True, get_support_data=True)

print("--- Loaded tplot variables ---")
print(pt.tplot_names())

11-Nov-25 13:25:41: Local file found: /mnt/j/observation_data//ergsc/satellite/erg/pwe/efd/l2/E64Hz/2022/09/erg_pwe_efd_l2_E64Hz_dsi_20220901_v01_02.cdf


 
 
**************************************************************************
['Exploration of Energization and Radiation in Geospace (ERG) Plasma Wave Experiment (PWE) Electric Field Detector (EFD) Level 2 waveform data in DSI Coordinate System']

Information about ERG PWE EFD

PI:  ['Yoshiya Kasahara']
Affiliation:  ['Kanazawa University']

RoR of ERG project common: https://ergsc.isee.nagoya-u.ac.jp/data_info/rules_of_the_road.shtml.en
RoR of PWE/EFD: https://ergsc.isee.nagoya-u.ac.jp/mw/index.php/ErgSat/Pwe/Efd

Contact: erg_pwe_info at isee.nagoya-u.ac.jp
**************************************************************************


11-Nov-25 13:25:42: Local file found: /mnt/j/observation_data//ergsc/satellite/erg/mgf/l2/64hz/2022/09/erg_mgf_l2_64hz_dsi_2022090121_v04.05.cdf
11-Nov-25 13:25:43: Local file found: /mnt/j/observation_data//ergsc/satellite/erg/mgf/l2/64hz/2022/09/erg_mgf_l2_64hz_dsi_2022090122_v04.05.cdf
11-Nov-25 13:25:43: Local file found: /mnt/j/observation_data//ergsc/satellite/erg/mgf/l2/64hz/2022/09/erg_mgf_l2_64hz_dsi_2022090123_v04.05.cdf


 
**************************************************************************
['Exploration of Energization and Radiation in Geospace (ERG) Magnetic Field Experiment (MGF) Level 2 64 Hz resolution magnetic field data']

Information about ERG MGF

PI:  ['Ayako Matsuoka']
Affiliation:  ['Data Analysis Center for Geomagnetism and Space Magnetism, Graduate School of Science, Kyoto University, Kitashirakawa-Oiwake Cho, Sakyo-ku Kyoto 606-8502, Japan']

RoR of ERG project common: https://ergsc.isee.nagoya-u.ac.jp/data_info/rules_of_the_road.shtml.en
RoR of MGF L2: https://ergsc.isee.nagoya-u.ac.jp/mw/index.php/ErgSat/Mgf
Contact: erg_mgf_info at isee.nagoya-u.ac.jp
**************************************************************************
--- Loaded tplot variables ---
0 : erg_pwe_efd_l2_E64Hz_dsi_Epoch
1 : erg_pwe_efd_l2_E64Hz_dsi_Ex_waveform
2 : erg_pwe_efd_l2_E64Hz_dsi_Ey_waveform
3 : erg_pwe_efd_l2_E64Hz_dsi_time_offsets
4 : erg_pwe_efd_l2_E64Hz_dsi_ti_corrected
5 : erg_pwe_efd_l2_E64Hz_d

In [28]:
time_range_T    = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]

E64_data_dsi_x  = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_Ex_waveform'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E64_data_dsi_y  = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_Ey_waveform'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E64_data_dsi_quality_flag = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_quality_flag'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

import xarray as xr
import numpy as np

# QF = 0 の時間を抽出
bad_times = E64_data_dsi_quality_flag.time.where(E64_data_dsi_quality_flag != 0, drop=True)

# 各電場データに対して、時間が bad_times に近いところを NaN に置換
def mask_by_quality(data, qf_times, tol='0.01s'):
    mask = xr.full_like(data, True, dtype=bool)
    # bad_times に含まれる時刻に最も近い時間をマスク
    for t in qf_times.values:
        near = np.abs(data.time - t) < np.timedelta64(int(np.float64(np.timedelta64(np.datetime64(t + np.timedelta64(10, 'ms')) - t))), 'ms')
        mask.loc[dict(time=data.time[near])] = False
    return data.where(mask, np.nan)

# 実際には単純な条件で十分
E64_data_dsi_x_qf = E64_data_dsi_x.where(E64_data_dsi_quality_flag.interp(time=E64_data_dsi_x.time, method='nearest', kwargs={'fill_value': 'extrapolate'}) == 0, np.nan)
E64_data_dsi_y_qf = E64_data_dsi_y.where(E64_data_dsi_quality_flag.interp(time=E64_data_dsi_y.time, method='nearest', kwargs={'fill_value': 'extrapolate'}) == 0, np.nan)

ds_E64_dsi = xr.Dataset({
    'E64_dsi_x': E64_data_dsi_x_qf,
    'E64_dsi_y': E64_data_dsi_y_qf
})

ds_E64_dsi  = ds_E64_dsi.dropna(dim='time', how='all')

In [35]:
B64_data_dsi    = pt.data_quants['erg_mgf_l2_mag_64hz_dsi'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B64_data_dsi_quality_flag   = pt.data_quants['erg_mgf_l2_quality_64hz'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

qf_B, B64 = xr.align(B64_data_dsi_quality_flag[:, 3], B64_data_dsi, join='inner')

B64_data_dsi_qf = xr.where(qf_B <= 21, B64, np.nan)

ds_B64_dsi  = xr.Dataset({
    'B64_dsi_x':    B64_data_dsi_qf[:, 0],
    'B64_dsi_y':    B64_data_dsi_qf[:, 1],
    'B64_dsi_z':    B64_data_dsi_qf[:, 2]
})

ds_B64_dsi  = ds_B64_dsi.dropna(dim='time', how='all')

In [12]:
def split_by_gap(ds, time_dim='time', gap_thr=np.timedelta64(30, 's'),
                 prefix='ds_8_gsm_seg'):
    t = ds[time_dim].values
    if t.size == 0:
        return []

    # 先頭はギャップなし。以降で gap_thr を超えたら新セグ開始。
    gaps = np.r_[False, (t[1:] - t[:-1]) > gap_thr]      # shape (Nt,)
    seg_id = np.cumsum(gaps)                              # 0,0,0,1,1,2,...

    ds_tagged = ds.assign_coords(_seg=(time_dim, seg_id))
    segs = [g.drop_vars('_seg') for _, g in ds_tagged.groupby('_seg')]

    return segs

In [32]:
ds_E64_dsi_segs = split_by_gap(ds_E64_dsi, gap_thr=np.timedelta64(8, 's'))
print(len(ds_E64_dsi_segs))

5


In [36]:
ds_B64_dsi_segs = split_by_gap(ds_B64_dsi, gap_thr=np.timedelta64(8, 's'))
print(len(ds_B64_dsi_segs))

1


# Wavelet analysis

In [37]:
import os
import sys
import importlib
import numpy as np
import matplotlib.pyplot as plt
import pyspedas as psp
import pytplot as pt
import pywt

sys.path.append("..")
import module_handmade.tdwavelet_themis as tw
importlib.reload(tw)

vars_E64    = ['E64_dsi_x','E64_dsi_y']
ds_E64_dsi_cwt_seg0 = tw.cwt_from_dataset(ds_E64_dsi_segs[0], dt=1/64, s0=2, dj=1/32, variables=vars_E64)
ds_E64_dsi_cwt_seg1 = tw.cwt_from_dataset(ds_E64_dsi_segs[1], dt=1/64, s0=2, dj=1/32, variables=vars_E64)
ds_E64_dsi_cwt_seg2 = tw.cwt_from_dataset(ds_E64_dsi_segs[2], dt=1/64, s0=2, dj=1/32, variables=vars_E64)
ds_E64_dsi_cwt_seg3 = tw.cwt_from_dataset(ds_E64_dsi_segs[3], dt=1/64, s0=2, dj=1/32, variables=vars_E64)
ds_E64_dsi_cwt_seg4 = tw.cwt_from_dataset(ds_E64_dsi_segs[4], dt=1/64, s0=2, dj=1/32, variables=vars_E64)

vars_B64    = ['B64_dsi_x','B64_dsi_y', 'B64_dsi_z']
ds_B64_dsi_cwt_seg0 = tw.cwt_from_dataset(ds_B64_dsi_segs[0], dt=1/64, s0=2, dj=1/32, variables=vars_B64)

print(ds_E64_dsi_cwt_seg3)
print(ds_B64_dsi_cwt_seg0)

<xarray.Dataset> Size: 1GB
Dimensions:        (freq: 342, time: 208225)
Coordinates:
  * freq           (freq) float32 1kB 32.0 31.31 30.64 ... 0.02026 0.01983
  * time           (time) datetime64[ns] 2MB 2022-09-01T22:05:53.618647040 .....
Data variables:
    E64_dsi_x_cwt  (time, freq) float64 570MB 4.966e-06 nan nan ... nan nan nan
    E64_dsi_x_coi  (time) float64 2MB 32.0 32.0 32.0 30.17 ... 32.0 32.0 32.0
    E64_dsi_y_cwt  (time, freq) float64 570MB 0.0002568 nan nan ... nan nan nan
    E64_dsi_y_coi  (time) float64 2MB 32.0 32.0 32.0 30.17 ... 32.0 32.0 32.0
<xarray.Dataset> Size: 7GB
Dimensions:        (freq: 397, time: 691940)
Coordinates:
  * freq           (freq) float32 2kB 32.05 31.36 30.69 ... 0.006166 0.006034
  * time           (time) datetime64[ns] 6MB 2022-09-01T21:00:00.001522176 .....
Data variables:
    B64_dsi_x_cwt  (time, freq) float64 2GB 229.8 nan nan nan ... nan nan nan
    B64_dsi_x_coi  (time) float64 6MB 32.05 32.05 32.05 ... 32.05 32.05 32.05
    B64_dsi

In [38]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm

# ---- セグメント連結（freq合わせ）----
def concat_cwt_segments(dsets, var):
    # dsets をリストに正規化
    if isinstance(dsets, xr.Dataset):
        dsets = [dsets]
    elif isinstance(dsets, (str, bytes)):
        raise TypeError("dsets は Dataset のリストにして")

    das = []
    for ds in dsets:
        if ds is None or not isinstance(ds, xr.Dataset):
            continue
        if var in ds.data_vars:
            das.append(ds[var])

    if not das:
        return None, None

    pow_cat = xr.concat(das, dim="time").sortby("time")

    coi_name = var.replace("_cwt", "_coi")
    coi_list = []
    for ds in dsets:
        if isinstance(ds, xr.Dataset) and coi_name in ds.data_vars:
            coi_list.append(ds[coi_name])
    coi_cat = xr.concat(coi_list, dim="time").sortby("time") if coi_list else None
    return pow_cat, coi_cat

# ---- 1面描画：外でax/caxを用意する ----
def plot_cwt_on_ax(ax, da_pow, da_coi=None, t0=None, minutes=5,
                   zrange=(1e-6, 1e3), yrange=(1e-2, 4.0),
                   cmap="turbo", label_left="", unit_right=""):
    # 時間切り出し
    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        da = da_pow.sel(time=slice(t0, t1))
        coi = da_coi.sel(time=slice(t0, t1)) if da_coi is not None else None
        #ax.set_xlim(t0, t1)
    else:
        da, coi = da_pow, da_coi
    if da.time.size == 0: return None, None

    T = mdates.date2num(da.time.values)
    F = da.freq.values
    Z = da.values.astype(float)

    # COIマスク（低周波側をNaN）
    if coi is not None:
        C = coi.values[:, None]
        Z = np.where(F[None, :] < C, np.nan, Z)

    # メッシュ
    Tm = np.tile(T, (F.size, 1)).T
    Fm = np.tile(F, (T.size, 1))

    pcm = ax.pcolormesh(Tm, Fm, Z, shading="auto",
                        norm=LogNorm(vmin=zrange[0], vmax=zrange[1]), cmap=cmap)

    ax.minorticks_on()
    ax.set_yscale("log")
    ax.set_ylim(yrange[0], yrange[1])
    ax.set_ylabel(f"{label_left}\n[Hz]")

    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype="datetime64[ns]")))

    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))   # 1分刻み
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.tick_params(axis="x", rotation=0)


    # 右側カラーバー
    cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    cb = plt.colorbar(pcm, cax=cax)
    cb.set_label(f"{unit_right}")
    return pcm, cb


In [40]:
dsets_E64 = [ds_E64_dsi_cwt_seg0, ds_E64_dsi_cwt_seg1, ds_E64_dsi_cwt_seg2, ds_E64_dsi_cwt_seg3, ds_E64_dsi_cwt_seg4]
targets = [
    ("E64_dsi_x_cwt", r"$E_{x}$ (DSI)", "[(mV/m)$^2$/Hz]"),
    ("E64_dsi_y_cwt", r"$E_{y}$ (DSI)", "[(mV/m)$^2$/Hz]")
]
joined = {}
for v, _, _ in targets:
    da, coi = concat_cwt_segments(dsets_E64, v)
    da = da.sortby('freq')
    if da is not None: joined[v] = (da.sortby("freq"), coi)

time_windows = [
    np.datetime64('2022-09-01T21:00') + np.timedelta64(5, 'm')*n
    for n in range(36)
]

for t0 in time_windows:
    fig, axes = plt.subplots(len(targets), 1, figsize=(10, 4), sharex=True)
    for ax, (v, ylab, unit) in zip(axes, targets):
        if v not in joined: continue
        da, coi = joined[v]
        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
                       zrange=(1e-6, 1e3), yrange=(1E-2, 32.0),
                       cmap="turbo", label_left=ylab, unit_right=unit)

    axes[-1].set_xlabel("time")
    fig.tight_layout()

    if os.path.isdir(path_base_save_plot):
        t0_str = str(t0)
        fn_time = t0_str.replace(':', '').replace('T', '_')
        fig_path = os.path.join(path_base_save_plot, f'E_fields_dsi_cwt_{fn_time}.png')
        print(fig_path)
        fig.savefig(fig_path)
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)

/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/E_fields_dsi_cwt_2022-09-01_2100.png
/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/E_fields_dsi_cwt_2022-09-01_2105.png
/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/E_fields_dsi_cwt_2022-09-01_2110.png
/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/E_fields_dsi_cwt_2022-09-01_2115.png
/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/E_fields_dsi_cwt_2022-09-01_2120.png
/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/E_fields_dsi_cwt_2022-09-01_2125.png
/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/E_fields_dsi_cwt_2022-09-01_2130.png
/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/E_fields_dsi_cwt_2022-09-01_2135.png
/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/E_fields_dsi_cwt_2022-09-01_2140.png
/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/E_fields_dsi_cwt_2022-09-01_2145.png
/mnt/j/KAW_observation/E_B_rat

In [41]:
dsets_B64 = [ds_B64_dsi_cwt_seg0]
targets = [
    ("B64_dsi_x_cwt", r"$B_{x}$ (DSI)", "[(nT)$^2$/Hz]"),
    ("B64_dsi_y_cwt", r"$B_{y}$ (DSI)", "[(nT)$^2$/Hz]"),
    ("B64_dsi_z_cwt", r"$B_{z}$ (DSI)", "[(nT)$^2$/Hz]")
]
joined = {}
for v, _, _ in targets:
    da, coi = concat_cwt_segments(dsets_B64, v)
    da = da.sortby('freq')
    if da is not None: joined[v] = (da.sortby("freq"), coi)

time_windows = [
    np.datetime64('2022-09-01T21:00') + np.timedelta64(5, 'm')*n
    for n in range(36)
]

for t0 in time_windows:
    fig, axes = plt.subplots(len(targets), 1, figsize=(10, 5), sharex=True)
    for ax, (v, ylab, unit) in zip(axes, targets):
        if v not in joined: continue
        da, coi = joined[v]
        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
                       zrange=(1e-6, 1e3), yrange=(1E-2, 32.0),
                       cmap="turbo", label_left=ylab, unit_right=unit)

    axes[-1].set_xlabel("time")
    fig.tight_layout()

    if os.path.isdir(path_base_save_plot):
        t0_str = str(t0)
        fn_time = t0_str.replace(':', '').replace('T', '_')
        fig_path = os.path.join(path_base_save_plot, f'B_fields_dsi_cwt_{fn_time}.png')
        print(fig_path)
        fig.savefig(fig_path)
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)

/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/B_fields_dsi_cwt_2022-09-01_2100.png
/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/B_fields_dsi_cwt_2022-09-01_2105.png
/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/B_fields_dsi_cwt_2022-09-01_2110.png
/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/B_fields_dsi_cwt_2022-09-01_2115.png
/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/B_fields_dsi_cwt_2022-09-01_2120.png
/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/B_fields_dsi_cwt_2022-09-01_2125.png
/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/B_fields_dsi_cwt_2022-09-01_2130.png
/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/B_fields_dsi_cwt_2022-09-01_2135.png
/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/B_fields_dsi_cwt_2022-09-01_2140.png
/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330/B_fields_dsi_cwt_2022-09-01_2145.png
/mnt/j/KAW_observation/E_B_rat

# 調査時刻におけるPSDのMedianを抽出

In [43]:
dsets_E64 = [ds_E64_dsi_cwt_seg0, ds_E64_dsi_cwt_seg1, ds_E64_dsi_cwt_seg2, ds_E64_dsi_cwt_seg3, ds_E64_dsi_cwt_seg4]
targets = [
    ("E64_dsi_x_cwt", r"$E_{x}$ (DSI)", "[(mV/m)$^2$/Hz]"),
    ("E64_dsi_y_cwt", r"$E_{y}$ (DSI)", "[(mV/m)$^2$/Hz]")
]
joined = {}
for v, _, _ in targets:
    da, coi = concat_cwt_segments(dsets_E64, v)
    da = da.sortby('freq')
    if da is not None: joined[v] = (da.sortby("freq"), coi)

da_E64_dsi_x_cwt    = joined['E64_dsi_x_cwt']
da_E64_dsi_y_cwt    = joined['E64_dsi_y_cwt']

In [44]:
dsets_B64 = [ds_B64_dsi_cwt_seg0]
targets = [
    ("B64_dsi_x_cwt", r"$B_{x}$ (DSI)", "[(nT)$^2$/Hz]"),
    ("B64_dsi_y_cwt", r"$B_{y}$ (DSI)", "[(nT)$^2$/Hz]"),
    ("B64_dsi_z_cwt", r"$B_{z}$ (DSI)", "[(nT)$^2$/Hz]")
]
joined = {}
for v, _, _ in targets:
    da, coi = concat_cwt_segments(dsets_B64, v)
    da = da.sortby('freq')
    if da is not None: joined[v] = (da.sortby("freq"), coi)

da_B64_dsi_x_cwt    = joined['B64_dsi_x_cwt']
da_B64_dsi_y_cwt    = joined['B64_dsi_y_cwt']
da_B64_dsi_z_cwt    = joined['B64_dsi_z_cwt']

In [52]:
import numpy as np
import xarray as xr

noise_t0 = np.datetime64('2022-09-01T21:30:00')
noise_t1 = np.datetime64('2022-09-01T22:00:00')

pairs = [
    ('E64_dsi_x_cwt', da_E64_dsi_x_cwt),
    ('E64_dsi_y_cwt', da_E64_dsi_y_cwt),
    ('B64_dsi_x_cwt', da_B64_dsi_x_cwt),
    ('B64_dsi_y_cwt', da_B64_dsi_y_cwt),
    ('B64_dsi_z_cwt', da_B64_dsi_z_cwt),
]

noise_da_dict = {}

for name, da in pairs:
    # (CWT, COI) のタプルなら CWT だけ使う
    if isinstance(da, tuple):
        da = da[0]
    if not isinstance(da, xr.DataArray):
        continue

    da = da.sortby('time').sel(time=slice(noise_t0, noise_t1))
    if da.sizes.get('time', 0) == 0:
        continue

    # 複素ならパワーに変換
    if np.iscomplexobj(da.data):
        da = (da.real**2 + da.imag**2)

    # 時間方向の NaN 無視メディアン
    med = da.median(dim='time', skipna=True)  # (freq,)
    noise_da_dict[name] = med

# 各 DataArray は独立（freq 軸も個別）
for k, v in noise_da_dict.items():
    print(f"{k}: {v.sizes}, freq range = [{v.freq.min().item():.3f}, {v.freq.max().item():.3f}]")

# まとめて返すなら dict のまま利用
noise_da_dict



E64_dsi_x_cwt: Frozen({'freq': 342}), freq range = [0.020, 32.000]
E64_dsi_y_cwt: Frozen({'freq': 342}), freq range = [0.020, 32.000]
B64_dsi_x_cwt: Frozen({'freq': 397}), freq range = [0.006, 32.051]
B64_dsi_y_cwt: Frozen({'freq': 397}), freq range = [0.006, 32.051]
B64_dsi_z_cwt: Frozen({'freq': 397}), freq range = [0.006, 32.051]


{'E64_dsi_x_cwt': <xarray.DataArray 'E64_dsi_x_cwt' (freq: 342)> Size: 3kB
 array([3.26499511e-02, 3.07126874e-02, 2.85164639e-02, 2.53792051e-02,
        2.20330954e-02, 1.86846964e-02, 1.57033112e-02, 2.32106764e-02,
        1.21334977e-02, 9.84965591e-03, 9.17930435e-03, 9.80154332e-03,
        8.41013249e-03, 6.87955250e-03, 7.60232448e-03, 6.11798628e-03,
        5.99194644e-03, 5.59487729e-03, 5.42617240e-03, 5.28511894e-03,
        4.93290881e-03, 4.61085979e-03, 4.68246592e-03, 5.52558992e-03,
        4.31445776e-03, 4.22499189e-03, 4.20199009e-03, 4.43199417e-03,
        4.26800107e-03, 4.15430171e-03, 3.94277787e-03, 3.91352479e-03,
        6.15829322e-03, 3.75815178e-03, 3.61267850e-03, 3.54998629e-03,
        3.42512643e-03, 3.40959721e-03, 3.28486215e-03, 3.80925555e-03,
        2.95121688e-03, 2.74850812e-03, 2.61024269e-03, 2.47280090e-03,
        2.72115413e-03, 2.23626732e-03, 2.32657394e-03, 1.98515272e-03,
        1.88378606e-03, 1.79793267e-03, 1.69815123e-03, 1.660

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

fig, ax = plt.subplots(figsize=(8, 6))

for name, da in noise_da_dict.items():
    prefix = name.split('_')[0]   # 'E64' or 'B64'
    comp   = name.split('_')[2]   # 'x', 'y', 'z'
    label  = f"${prefix[0]}_{comp}$"
    ax.loglog(da['freq'], da, label=label)

ax.minorticks_on()
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('Median PSD')
ax.set_title('Median 21:30–22:00  (E: (mV/m)$^2$/Hz,  B: nT$^2$/Hz)')
ax.grid(True, which='both', ls=':')
ax.legend(ncol=3)
ax.set_xlim(1e-2, 32)
ax.set_ylim(1e-8, 5e1)
plt.tight_layout()

# 保存 or 表示
if os.path.isdir(path_base_save_plot):
    fn = f"median_bs_{str(noise_t0).replace(':','')}_{str(noise_t1).replace(':','')}.png"
    fig.savefig(os.path.join(path_base_save_plot, fn), dpi=300, bbox_inches='tight')
    plt.close(fig)
else:
    plt.show()
    plt.close(fig)


In [55]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os

# ---- 入力 ----
pairs = [
    ('E64_dsi_x_cwt', da_E64_dsi_x_cwt),
    ('E64_dsi_y_cwt', da_E64_dsi_y_cwt),
    ('B64_dsi_x_cwt', da_B64_dsi_x_cwt),
    ('B64_dsi_y_cwt', da_B64_dsi_y_cwt),
    ('B64_dsi_z_cwt', da_B64_dsi_z_cwt),
]
t_all_start = np.datetime64('2022-09-01T23:30:00')
t_all_end   = np.datetime64('2022-09-02T00:00:00')
step        = np.timedelta64(5, 'm')  # 5分
outdir      = path_base_save_plot  # 既存の保存先を使用

# 事前に CWT 本体のみ取り出し、timeでソート
pairs_sorted = []
for name, da in pairs:
    if isinstance(da, tuple):
        da = da[0]
    if isinstance(da, xr.DataArray):
        pairs_sorted.append((name, da.sortby('time')))

def compute_median_dict(pairs_sorted, t0, t1):
    d = {}
    for name, da in pairs_sorted:
        sub = da.sel(time=slice(t0, t1))
        if sub.sizes.get('time', 0) == 0:
            continue
        if np.iscomplexobj(sub.data):
            sub = (sub.real**2 + sub.imag**2)
        d[name] = sub.median(dim='time', skipna=True)  # (freq,)
    return d

def plot_median_dict(mdict, t0, t1, outdir=None):
    fig, ax = plt.subplots(figsize=(8, 6))
    for name, med in mdict.items():
        prefix = name.split('_')[0]  # 'E64' or 'B64'
        comp   = name.split('_')[2]  # 'x','y','z'
        label  = f"${prefix[0]}_{comp}$"
        ax.loglog(med['freq'], med, label=label)

    ax.minorticks_on()
    ax.set_xlabel('Frequency [Hz]')
    ax.set_ylabel('Median PSD')
    ax.set_title(f"Median {str(t0)[11:16]}–{str(t1)[11:16]}  (E: (mV/m)$^2$/Hz,  B: nT$^2$/Hz)")
    ax.grid(True, which='both', ls=':')
    ax.legend(ncol=3)
    ax.set_xlim(1e-2, 32)
    ax.set_ylim(1e-8, 5e1)
    plt.tight_layout()

    if outdir and os.path.isdir(outdir):
        fn = f"median_bs_{str(t0).replace(':','')}_{str(t1).replace(':','')}.png"
        fig.savefig(os.path.join(outdir, fn), dpi=300, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)

# ---- 5分窓でループ ----
t_starts = np.arange(t_all_start, t_all_end, step)  # 21:00, 21:05, ..., 23:25
for t0 in t_starts:
    t1 = t0 + step
    mdict = compute_median_dict(pairs_sorted, t0, t1)
    if not mdict:  # その窓でデータ無し
        continue
    plot_median_dict(mdict, t0, t1, outdir=outdir)
